# 🏥 CareFlow — Notebook 18: Hospital Census & Operations
## 250-Bed Ward Occupancy, CMS OP-18 Emergency Dwell & EVS Turnover Telemetry

**Author:** CareFlow Hospital Operations Command  
**Standards:** CMS OP-18 (ED Length of Stay), Joint Commission Emergency Management Standards  
**Dataset:** `careflow_analytics/output/stg_ehr_events.parquet`

---

### Objectives:
1. Model real-time bed census across 7 clinical units (ED Zone A/B, Trauma Resus, MICU, CCU, Step-Down, Med-Surg).
2. Measure Environmental Services (EVS) bed turnover distributions.
3. Calculate CMS OP-18 and SEP-1 hospital compliance scores.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('dark_background')

units = [
    {"Unit": "ED Zone A (Acute)", "Beds": 32, "Occupied": 29},
    {"Unit": "ED Zone B (Subacute)", "Beds": 28, "Occupied": 22},
    {"Unit": "Trauma Resuscitation", "Beds": 8, "Occupied": 6},
    {"Unit": "Medical ICU (MICU)", "Beds": 24, "Occupied": 23},
    {"Unit": "Coronary Care (CCU)", "Beds": 18, "Occupied": 14},
    {"Unit": "Step-Down Unit", "Beds": 40, "Occupied": 37},
    {"Unit": "Med-Surg East", "Beds": 50, "Occupied": 42},
    {"Unit": "Med-Surg West", "Beds": 50, "Occupied": 45},
]

census_df = pd.DataFrame(units)
census_df['Occupancy %'] = (census_df['Occupied'] / census_df['Beds']) * 100
census_df['Status'] = np.where(census_df['Occupancy %'] >= 90.0, 'Critical',
                      np.where(census_df['Occupancy %'] >= 85.0, 'Advisory', 'Nominal'))

print("Hospital Bed Census & Saturation Leaderboard:")
census_df


In [ ]:
# Visualize Bed Saturation
plt.figure(figsize=(10, 5))
colors = ['#f43f5e' if s == 'Critical' else '#f59e0b' if s == 'Advisory' else '#10b981' for s in census_df['Status']]
bars = plt.barh(census_df['Unit'], census_df['Occupancy %'], color=colors, alpha=0.85)

plt.axvline(85, color='#f59e0b', linestyle=':', label='85% Advisory')
plt.axvline(90, color='#f43f5e', linestyle='--', label='90% Critical Saturation')

for bar in bars:
    w = bar.get_width()
    plt.text(w + 1, bar.get_y() + bar.get_height()/2, f"{w:.1f}%", va='center', color='#fff', fontweight='bold')

plt.gca().invert_yaxis()
plt.title("Facility Ward Bed Occupancy & Capacity Monitoring")
plt.xlabel("Bed Occupancy %")
plt.xlim(0, 115)
plt.legend()
plt.tight_layout()
plt.show()
